# 04 - Educational Guide: Semi-Supervised Learning for Segmentation

This notebook provides a broader tutorial on semi-supervised learning concepts, using interactive demos to build intuition for the techniques used in our self-training pipeline.

**What you will learn:**
1. The semi-supervised learning landscape and taxonomy
2. Why confirmation bias is dangerous (toy 2D demo)
3. How curriculum learning helps (progressive difficulty)
4. Practical checklist for applying self-training to new datasets
5. Key references for further reading

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyArrowPatch

%matplotlib inline

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11
rng = np.random.RandomState(42)

## 1. Semi-Supervised Learning Landscape

Semi-supervised learning (SSL) sits between fully supervised (all labels) and unsupervised (no labels) learning. The key idea: **use a small amount of labeled data together with a large amount of unlabeled data.**

### Taxonomy of Approaches

```
Semi-Supervised Learning
|
+-- Pseudo-Labeling / Self-Training
|   |-- Classic self-training (Lee, 2013)
|   |-- Noisy Student (Xie et al., 2020)
|   +-- FixMatch (Sohn et al., 2020)          <-- Our approach builds on this
|
+-- Consistency Regularization
|   |-- Pi-Model (Laine & Aila, 2017)
|   |-- Temporal Ensembling
|   +-- Mean Teacher (Tarvainen & Valpola, 2017)  <-- We use this
|
+-- Hybrid Approaches
|   |-- MixMatch (Berthelot et al., 2019)
|   |-- FlexMatch (Zhang et al., 2021)
|   +-- Cross Pseudo Supervision (Chen et al., 2021)
|
+-- Graph-Based / Generative
    |-- Label propagation
    +-- VAE/GAN-based approaches
```

Our pipeline combines **pseudo-labeling** (teacher generates labels) with **consistency regularization** (student and teacher should agree) and **curriculum learning** (easy-to-hard progression).

In [ ]:
# Visualize the learning paradigm spectrum
fig, ax = plt.subplots(figsize=(14, 3))

paradigms = [
    (0.0, "Unsupervised\n(no labels)", "#e74c3c"),
    (0.15, "Self-Supervised\n(pretext tasks)", "#e67e22"),
    (0.45, "Semi-Supervised\n(few labels +\nunlabeled)", "#f1c40f"),
    (0.75, "Weakly-Supervised\n(noisy/partial\nlabels)", "#2ecc71"),
    (1.0, "Fully-Supervised\n(all labels)", "#3498db"),
]

for pos, label, color in paradigms:
    ax.plot(pos, 0, "o", markersize=20, color=color, zorder=5)
    y_offset = 0.15 if pos in [0.15, 0.75] else -0.25
    ax.text(pos, y_offset, label, ha="center", va="center", fontsize=9, fontweight="bold")

# Arrow
ax.annotate(
    "",
    xy=(1.05, 0),
    xytext=(-0.05, 0),
    arrowprops={"arrowstyle": "->", "linewidth": 2, "color": "gray"},
)
ax.text(0.5, -0.55, "Amount of Supervision", ha="center", fontsize=12, style="italic")

# Highlight our zone
ax.axvspan(0.3, 0.6, alpha=0.15, color="#f1c40f")
ax.text(0.45, 0.45, "Our approach", ha="center", fontsize=10, color="#d4ac0d", fontweight="bold")

ax.set_xlim(-0.1, 1.15)
ax.set_ylim(-0.65, 0.6)
ax.axis("off")
ax.set_title("The Learning Paradigm Spectrum", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. The Confirmation Bias Problem

Confirmation bias is the central challenge of pseudo-labeling: **the model reinforces its own mistakes.** If the model confidently mislabels a region, it trains on that mistake and becomes even more confident.

Let's demonstrate this with a toy 2D classification problem.

In [ ]:
# Generate a 2D dataset with a non-linear boundary
n_labeled = 20
n_unlabeled = 200

# True boundary: a curve
def true_boundary(x):
    return 0.3 * np.sin(3 * x) + 0.5


# Generate points
all_x = rng.uniform(0, 1, n_labeled + n_unlabeled)
all_y = rng.uniform(0, 1, n_labeled + n_unlabeled)
all_labels = (all_y > true_boundary(all_x)).astype(int)

# Split into labeled and unlabeled
labeled_x, labeled_y = all_x[:n_labeled], all_y[:n_labeled]
labeled_labels = all_labels[:n_labeled]
unlabeled_x, unlabeled_y = all_x[n_labeled:], all_y[n_labeled:]
unlabeled_true_labels = all_labels[n_labeled:]


def simple_classifier(x, y, labeled_x, labeled_y, labeled_labels, k=3):
    """K-nearest neighbors classifier."""
    predictions = []
    confidences = []
    for xi, yi in zip(x, y):
        dists = np.sqrt((labeled_x - xi) ** 2 + (labeled_y - yi) ** 2)
        nn_idx = np.argsort(dists)[:k]
        nn_labels = labeled_labels[nn_idx]
        vote = nn_labels.mean()
        pred = int(vote > 0.5)
        conf = max(vote, 1 - vote)
        predictions.append(pred)
        confidences.append(conf)
    return np.array(predictions), np.array(confidences)


# Round 0: Classify with only labeled data
pred_r0, conf_r0 = simple_classifier(unlabeled_x, unlabeled_y, labeled_x, labeled_y, labeled_labels)
acc_r0 = (pred_r0 == unlabeled_true_labels).mean()

# Round 1 (naive): Add ALL pseudo-labels and reclassify
aug_x = np.concatenate([labeled_x, unlabeled_x])
aug_y = np.concatenate([labeled_y, unlabeled_y])
aug_labels = np.concatenate([labeled_labels, pred_r0])
pred_r1, conf_r1 = simple_classifier(unlabeled_x, unlabeled_y, aug_x, aug_y, aug_labels)
acc_r1 = (pred_r1 == unlabeled_true_labels).mean()

# Round 1 (filtered): Add only HIGH-confidence pseudo-labels
high_conf = conf_r0 >= 0.9
aug_x_filt = np.concatenate([labeled_x, unlabeled_x[high_conf]])
aug_y_filt = np.concatenate([labeled_y, unlabeled_y[high_conf]])
aug_labels_filt = np.concatenate([labeled_labels, pred_r0[high_conf]])
pred_r1_filt, conf_r1_filt = simple_classifier(unlabeled_x, unlabeled_y, aug_x_filt, aug_y_filt, aug_labels_filt)
acc_r1_filt = (pred_r1_filt == unlabeled_true_labels).mean()

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

x_line = np.linspace(0, 1, 100)

for ax, title, preds, accuracy in [
    (axes[0], f"Round 0: Labeled Only\n(Accuracy: {acc_r0:.1%})", pred_r0, acc_r0),
    (axes[1], f"Round 1: Naive Pseudo-Labels\n(Accuracy: {acc_r1:.1%})", pred_r1, acc_r1),
    (axes[2], f"Round 1: Filtered Pseudo-Labels\n(Accuracy: {acc_r1_filt:.1%})", pred_r1_filt, acc_r1_filt),
]:
    # Plot true boundary
    ax.plot(x_line, true_boundary(x_line), "k-", linewidth=2, label="True boundary")

    # Color unlabeled points by prediction
    correct = preds == unlabeled_true_labels
    ax.scatter(
        unlabeled_x[correct & (preds == 1)],
        unlabeled_y[correct & (preds == 1)],
        c="#3498db",
        alpha=0.4,
        s=15,
        label="Correct",
    )
    ax.scatter(
        unlabeled_x[correct & (preds == 0)],
        unlabeled_y[correct & (preds == 0)],
        c="#e74c3c",
        alpha=0.4,
        s=15,
    )
    ax.scatter(
        unlabeled_x[~correct],
        unlabeled_y[~correct],
        c="black",
        marker="x",
        s=30,
        label="Errors",
    )

    # Plot labeled points
    ax.scatter(
        labeled_x[labeled_labels == 1],
        labeled_y[labeled_labels == 1],
        c="#3498db",
        marker="s",
        s=60,
        edgecolors="black",
        linewidth=1.5,
        zorder=5,
        label="Labeled",
    )
    ax.scatter(
        labeled_x[labeled_labels == 0],
        labeled_y[labeled_labels == 0],
        c="#e74c3c",
        marker="s",
        s=60,
        edgecolors="black",
        linewidth=1.5,
        zorder=5,
    )

    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8, loc="upper left")

fig.suptitle(
    "Confirmation Bias: Naive vs Filtered Pseudo-Labeling",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

print("Observations:")
print(f"  Round 0 (labeled only):   {acc_r0:.1%} accuracy")
print(f"  Round 1 (naive, all PLs): {acc_r1:.1%} accuracy")
print(f"  Round 1 (filtered PLs):   {acc_r1_filt:.1%} accuracy")
print()
if acc_r1 <= acc_r0:
    print("  Naive pseudo-labeling HURT performance (confirmation bias).")
print("  Confidence filtering preserves or improves performance.")

## 3. Curriculum Learning Intuition

Curriculum learning (Bengio et al., 2009) is the idea that models learn better when training examples are presented in order of increasing difficulty --- just like how students learn: start with basics, then advance to harder material.

In our self-training context:
- **Easy examples**: Unlabeled volumes where the model is very confident (e.g., large, well-defined tumors)
- **Hard examples**: Ambiguous cases near decision boundaries (e.g., small enhancing tumors, subtle edema)

The curriculum threshold schedule naturally implements this: high thresholds in early rounds accept only easy cases, while lower thresholds in later rounds incorporate harder ones.

In [ ]:
# Demonstrate curriculum learning with progressively harder examples
n_samples = 500

# Generate samples with varying "difficulty" (distance to boundary)
samples_x = rng.uniform(0, 1, n_samples)
samples_y = rng.uniform(0, 1, n_samples)
boundary_y = true_boundary(samples_x)
distance_to_boundary = np.abs(samples_y - boundary_y)

# Difficulty: closer to boundary = harder
difficulty = 1.0 - np.clip(distance_to_boundary / 0.3, 0, 1)

# Simulate model confidence (inversely related to difficulty)
model_confidence = np.clip(0.5 + 0.5 * (1.0 - difficulty) + rng.normal(0, 0.05, n_samples), 0.5, 1.0)

# Curriculum thresholds
import math

thresholds = []
for r in range(4):
    t = r / 3
    threshold = 0.75 + (0.95 - 0.75) * 0.5 * (1.0 + math.cos(math.pi * t))
    thresholds.append(threshold)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for r, (ax, thresh) in enumerate(zip(axes, thresholds)):
    accepted = model_confidence >= thresh
    n_accepted = accepted.sum()

    ax.scatter(
        samples_x[~accepted],
        samples_y[~accepted],
        c="lightgray",
        s=8,
        alpha=0.5,
        label=f"Rejected ({(~accepted).sum()})",
    )
    scatter = ax.scatter(
        samples_x[accepted],
        samples_y[accepted],
        c=model_confidence[accepted],
        cmap="RdYlGn",
        vmin=0.5,
        vmax=1.0,
        s=15,
        alpha=0.7,
        label=f"Accepted ({n_accepted})",
    )
    ax.plot(x_line, true_boundary(x_line), "k-", linewidth=1.5)

    ax.set_title(f"Round {r}: tau={thresh:.2f}\n{n_accepted}/{n_samples} accepted", fontsize=10, fontweight="bold")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=7, loc="upper left")

fig.suptitle(
    "Curriculum Thresholding: Easy to Hard",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

print("Key insight:")
print("  Round 0: Only easy cases far from the boundary are accepted.")
print("  Round 3: Most cases are accepted, including harder boundary cases.")
print("  This prevents the model from learning wrong boundary locations early on.")

## 4. Defenses Against Confirmation Bias

Our pipeline uses multiple complementary strategies to combat confirmation bias:

| Defense | How It Works | Implementation |
|---------|-------------|----------------|
| **EMA Teacher** | Temporal smoothing of student weights | `theta_t = 0.999 * theta_t + 0.001 * theta_s` |
| **Confidence Filtering** | Reject uncertain pseudo-labels | Per-voxel, per-class threshold |
| **Curriculum Schedule** | Start strict, relax gradually | Cosine: 0.95 to 0.75 |
| **Per-Class Thresholds** | Stricter for hard classes | ET: +0.05 offset |
| **Strong Augmentation** | Student sees augmented input | Noise, gamma, contrast |
| **Loss Ramp-Up** | Gradually increase pseudo-label weight | Linear over 20 epochs |
| **Consistency Loss** | Encourage student-teacher agreement | MSE between logits |

In [ ]:
# Simulate the effect of each defense mechanism

n_rounds = 10  # More rounds to show long-term effects

def simulate_self_training(
    n_rounds=10,
    use_ema=False,
    use_filtering=False,
    use_curriculum=False,
    noise_rate=0.15,
):
    """Simulate self-training accuracy over rounds with different defenses."""
    accuracy = 0.80  # Starting accuracy
    accuracies = [accuracy]

    for r in range(n_rounds):
        # Pseudo-label noise depends on current accuracy
        pl_noise = noise_rate * (1 - accuracy)

        if use_ema:
            pl_noise *= 0.6  # EMA reduces noise by ~40%

        if use_filtering:
            if use_curriculum:
                t = r / max(n_rounds - 1, 1)
                threshold = 0.75 + (0.95 - 0.75) * 0.5 * (1 + math.cos(math.pi * t))
            else:
                threshold = 0.85
            # Filtering reduces noise but also reduces data utilization
            effective_noise = pl_noise * (1 - threshold)
            data_util = 1 - threshold + 0.5  # More data at lower thresholds
        else:
            effective_noise = pl_noise
            data_util = 1.0

        # Update accuracy: benefit from extra data minus noise penalty
        benefit = 0.005 * data_util * (1 - accuracy)  # Diminishing returns
        penalty = effective_noise * 0.3
        accuracy = accuracy + benefit - penalty
        accuracy = np.clip(accuracy, 0.5, 0.99)
        accuracies.append(accuracy)

    return accuracies


configs = [
    ("Naive (no defenses)", {"use_ema": False, "use_filtering": False}),
    ("+ Confidence Filtering", {"use_ema": False, "use_filtering": True}),
    ("+ EMA Teacher", {"use_ema": True, "use_filtering": True}),
    ("+ Curriculum (full pipeline)", {"use_ema": True, "use_filtering": True, "use_curriculum": True}),
]

fig, ax = plt.subplots(figsize=(12, 6))
colors_config = ["#e74c3c", "#f39c12", "#3498db", "#27ae60"]
linestyles = ["--", "-.", ":", "-"]

for (name, kwargs), color, ls in zip(configs, colors_config, linestyles):
    accs = simulate_self_training(**kwargs)
    ax.plot(range(len(accs)), accs, linewidth=2.5, label=name, color=color, linestyle=ls)

ax.axhline(y=0.80, color="gray", linestyle="--", alpha=0.5, label="Baseline")
ax.set_xlabel("Self-Training Round", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title(
    "Effect of Confirmation Bias Defenses on Self-Training",
    fontsize=14,
    fontweight="bold",
)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.72, 0.88)
plt.tight_layout()
plt.show()

print("Without defenses, self-training can DECREASE performance (confirmation bias).")
print("Each defense adds protection; the full pipeline maintains steady improvement.")

## 5. Practical Checklist for Applying Self-Training

If you want to apply self-training to a new segmentation task, here is a checklist of considerations.

### Prerequisites

- [ ] **Baseline model performance is reasonable.** Self-training amplifies existing model quality. If your baseline Dice is 0.50, self-training will produce poor pseudo-labels. Aim for baseline Dice > 0.70 before self-training.

- [ ] **Sufficient unlabeled data.** The unlabeled set should be at least 25-50% the size of the labeled set to provide meaningful benefit. More is better.

- [ ] **Same domain.** Unlabeled data should come from the same acquisition protocol, scanner, and patient population as labeled data. Domain shift between labeled and unlabeled sets will cause pseudo-label noise.

- [ ] **Clean validation set.** Keep a purely labeled validation set that is NEVER contaminated with pseudo-labels. This is your ground truth for fair evaluation.

### Hyperparameter Selection

| Parameter | Start With | Tune If... |
|-----------|-----------|------------|
| EMA decay | 0.999 | Teacher too noisy (decrease) or too slow (increase) |
| Initial threshold | 0.95 | Too few pseudo-labels accepted (decrease) |
| Final threshold | 0.75 | Too many noisy pseudo-labels (increase) |
| Pseudo-label weight | 0.5 | Pseudo-labels dominating (decrease) |
| Consistency weight | 0.1 | Student-teacher diverging (increase) |
| Ramp-up epochs | 20 | Early instability (increase) |
| Number of rounds | 4 | Diminishing returns (decrease) or continuing improvement (increase) |

### Common Pitfalls

1. **Using too low an initial threshold.** This admits noisy pseudo-labels before the teacher is reliable. Start conservative.

2. **Not using per-class thresholds.** If your task has classes with very different difficulty levels, a single global threshold will be too strict for easy classes and too lenient for hard ones.

3. **Contaminating the validation set.** Never include pseudo-labeled volumes in validation. This inflates metrics and gives a false sense of improvement.

4. **Ignoring class imbalance.** If pseudo-labels are biased toward the majority class, the model will learn this bias. Monitor per-class acceptance rates.

5. **Running too many rounds without checkpointing.** Always save per-round checkpoints. The best round may not be the last one.

In [ ]:
# Decision tree for when to use self-training

print("DECISION GUIDE: Should You Use Self-Training?")
print("=" * 55)
print()
print("  1. Do you have unlabeled data from the same domain?")
print("     NO  --> Consider transfer learning or data augmentation instead")
print("     YES --> Continue")
print()
print("  2. Is your baseline model reasonably good (Dice > 0.70)?")
print("     NO  --> Improve baseline first (more data, better architecture, tuning)")
print("     YES --> Continue")
print()
print("  3. Is unlabeled set >= 25% of labeled set?")
print("     NO  --> Benefit may be marginal; try anyway with 1-2 rounds")
print("     YES --> Continue")
print()
print("  4. Do you have GPU resources for 2-4x training time?")
print("     NO  --> Try 1 round of pseudo-labeling (still helps)")
print("     YES --> Full self-training pipeline recommended")
print()
print("  5. Are your classes balanced in difficulty?")
print("     YES --> Global threshold is fine")
print("     NO  --> Use per-class thresholds (stricter for hard classes)")

## 6. Beyond Self-Training: Other Semi-Supervised Approaches

Self-training is one of many semi-supervised approaches. Here is how alternatives compare:

| Method | Strengths | Weaknesses | Best For |
|--------|-----------|------------|----------|
| **Self-Training** (ours) | Simple, works with any model | Confirmation bias risk | When you have a good baseline |
| **Mean Teacher** | Stable pseudo-labels | Requires 2x model memory | When teacher stability matters |
| **FixMatch** | Strong augmentation + thresholding | Sensitive to threshold choice | Classification tasks |
| **CPS (Cross Pseudo Supervision)** | Two networks reduce confirmation bias | 2x compute, complex training | When diversity is critical |
| **MixMatch** | Data augmentation + mixing | Complex, many hyperparameters | When labeled data is very scarce |
| **Noisy Student** | Simple iterative training | No online teacher | Large-scale settings |

Our pipeline combines elements of Self-Training + Mean Teacher + FixMatch (strong augmentation) + curriculum learning.

## 7. Key References

### Foundational
1. **Lee (2013)** - "Pseudo-label: The simple and efficient semi-supervised learning method for deep neural networks." The original pseudo-labeling paper.
2. **Tarvainen & Valpola (2017)** - "Mean teachers are better role models." Introduced EMA teacher for semi-supervised learning.
3. **Bengio et al. (2009)** - "Curriculum learning." Foundational work on training with easy-to-hard ordering.

### Modern Methods
4. **Sohn et al. (2020)** - "FixMatch: Simplifying Semi-Supervised Learning with Consistency and Confidence." Unified pseudo-labeling + consistency with strong augmentation.
5. **Xie et al. (2020)** - "Self-training with Noisy Student improves ImageNet classification." Scaled self-training to ImageNet.
6. **Zhang et al. (2021)** - "FlexMatch: Boosting Semi-Supervised Learning with Curriculum Pseudo Labeling." Adaptive per-class thresholds.

### Medical Imaging
7. **Hatamizadeh et al. (2022)** - "Swin UNETR: Swin Transformers for Semantic Segmentation of Brain Tumors in MRI Images." The architecture we build on.
8. **Yu et al. (2019)** - "Uncertainty-Aware Self-Ensembling Model for Semi-Supervised 3D Left Atrium Segmentation." Uncertainty-weighted consistency for medical SSL.
9. **Chen et al. (2021)** - "Semi-supervised Semantic Segmentation with Cross Pseudo Supervision." Cross-network pseudo-label diversity.

### Surveys
10. **Yang et al. (2022)** - "A Survey on Deep Semi-supervised Learning." Comprehensive overview of modern SSL methods.

---

## Summary

Semi-supervised self-training is a practical technique for leveraging unlabeled data. The key principles are:

1. **Start from a good baseline.** The quality of pseudo-labels depends on the model's initial performance.
2. **Use confidence filtering.** Never train on uncertain predictions.
3. **Apply curriculum scheduling.** Easy examples first, hard examples later.
4. **Combine defenses.** EMA teacher + filtering + curriculum + consistency > any single defense.
5. **Monitor per-round metrics.** Stop if performance stops improving or starts degrading.

These principles apply beyond brain tumor segmentation --- to any task where labeled data is scarce but unlabeled data is available.